In [55]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler, StringIndexer, Imputer, StringIndexer, StandardScaler, OneHotEncoder
from pyspark.ml.classification import LogisticRegression, GBTClassifier, RandomForestClassifier, DecisionTreeClassifier
from pyspark.ml.tuning import ParamGridBuilder, TrainValidationSplit
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml import Pipeline

import mlflow
import mlflow.spark
from hyperopt import fmin, tpe, hp, SparkTrials, STATUS_OK
import numpy as np


from pyspark.sql.types import (
    StringType, IntegerType, LongType, FloatType, DoubleType, DecimalType,
    BooleanType, DateType, TimestampType
)


In [56]:
spark = SparkSession.builder.appName("ClassificationModelTrainer").getOrCreate()

In [57]:
class ClassificationModelTrainer:
    def __init__(self, train_data_path: str, test_data_path: str, spark: SparkSession):
        self.train_data_path = train_data_path
        self.test_data_path = test_data_path
        self.spark = spark

    def load_data(self):
        self.train_data = self.spark.read.csv(self.train_data_path, header=True, inferSchema=True)
        self.test_data = self.spark.read.csv(self.test_data_path, header=True, inferSchema=True)

    def identity_columns(self):
        fields = self.train_data.schema.fields
        self.label_col = 'booking_status_indexed'  # Assuming the label column is named
         # Categorical columns (strings)
        self.categorical_cols = [
            f.name for f in fields
            if isinstance(f.dataType, StringType) and f.name != self.label_col
        ]

        # Numerical columns (common numeric types)
        numeric_types = (IntegerType, LongType, FloatType, DoubleType, DecimalType)
        self.numerical_cols = [
            f.name for f in fields
            if isinstance(f.dataType, numeric_types) and f.name != self.label_col
        ]

        
    
    def create_pipeline(self, classifier):
        pipeline_stages = []

        # imputer
        imputer = Imputer(inputCols=self.numerical_cols, outputCols=[f"{col}_imputed" for col in self.numerical_cols])
        pipeline_stages.append(imputer)

        # string indexer and one-hot encoder for categorical features
        for col in self.categorical_cols:
            indexer = StringIndexer(inputCol=col, outputCol=f"{col}_indexed", handleInvalid='keep')
            pipeline_stages.append(indexer)

            encoder = OneHotEncoder(inputCol=f"{col}_indexed", outputCol=f"{col}_encoded" , dropLast=False)
            pipeline_stages.append(encoder)

        # vector assembler
        input_cols = [f"{col}_imputed" for col in self.numerical_cols] + [f"{col}_encoded" for col in self.categorical_cols]
        assembler = VectorAssembler(inputCols=input_cols, outputCol="features_unscales")
        pipeline_stages.append(assembler)

        # standard scaler
        scaler = StandardScaler(inputCol="features_unscales", outputCol="features")
        pipeline_stages.append(scaler)

        # add classifier
        pipeline_stages.append(classifier)

        self.pipeline = Pipeline(stages=pipeline_stages)

    def tune_hyperparameters(self):
        if isinstance(self.classifier, DecisionTreeClassifier):
            param_grid = (ParamGridBuilder()
                .addGrid(self.classifier.maxDepth, [5, 10])
                .addGrid(self.classifier.maxBins, [32, 64])
                .build())
        elif isinstance(self.classifier, RandomForestClassifier):
            param_grid = (ParamGridBuilder()
                .addGrid(self.classifier.numTrees, [50,100])
                .addGrid(self.classifier.maxDepth, [5, 10])
                .build())
        elif isinstance(self.classifier, GBTClassifier):
            param_grid = (ParamGridBuilder()
                .addGrid(self.classifier.maxDepth, [5, 10])
                .addGrid(self.classifier.maxBins, [32, 64])
                .addGrid(self.classifier.stepSize, [0.01, 0.1])
                .build())
        elif isinstance(self.classifier, LogisticRegression):
            param_grid = (ParamGridBuilder()
                .addGrid(self.classifier.regParam, [0.01, 0.1])
                .addGrid(self.classifier.elasticNetParam, [0.25, 0.75])
                .build())
        else:
            raise ValueError("Unsupported classifier type for hyperparameter tuning")

        tvs = TrainValidationSplit(estimator=self.pipeline,
                                  estimatorParamMaps=param_grid,
                                  evaluator=MulticlassClassificationEvaluator(labelCol=self.label_col, metricName="accuracy"),
                                  trainRatio=0.8)

        self.model = tvs.fit(self.train_data)

    def evaluate_model(self):
        predictions = self.model.transform(self.test_data)
        evaluator = MulticlassClassificationEvaluator(labelCol=self.label_col, predictionCol="prediction", metricName="accuracy")
        accuracy = evaluator.evaluate(predictions)
        print(f"Test Accuracy: {accuracy}")
        return accuracy
    
    def log_model_and_metrics(self,mlflow_run_name):
        mlflow.set_tracking_uri("http://localhost:5000")
        mlflow.set_experiment("booking_classification_experiment")
        #mlflow.pyspark.ml.autolog()
        with mlflow.start_run(run_name=mlflow_run_name):
            mlflow.spark.log_model(self.model.bestModel, "classificationmodel")
            mlflow.log_metric("accuracy", self.evaluate_model())
    
    def train_and_tune_model(self):
        self.load_data()
        self.identity_columns()
        self.create_pipeline(self.classifier)
        self.tune_hyperparameters()
        return self.model.bestModel
    
    def run_pipeline(self, mlflow_run_name):
        model = self.train_and_tune_model()
        self.log_model_and_metrics(mlflow_run_name)
        return model
    
    

In [ ]:
if __name__ == "__main__":
    train_data_path = "work/dataset/train_hotel_reservations.csv" #1 to 36249
    test_data_path = "work/dataset/test_hotel_reservations.csv"

    classifiers = [
        DecisionTreeClassifier(labelCol="booking_status_indexed", featuresCol="features"),
        RandomForestClassifier(labelCol="booking_status_indexed", featuresCol="features"),
        LogisticRegression(labelCol="booking_status_indexed", featuresCol="features"),
        GBTClassifier(labelCol="booking_status_indexed", featuresCol="features")
    ]

    for classifier in classifiers:
        trainer = ClassificationModelTrainer(train_data_path, test_data_path, spark)
        trainer.classifier = classifier
        trained_model = trainer.run_pipeline(f"model_{type(classifier).__name__}")

2026/03/13 00:12:27 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '5459478a0c5745efba7db1e8315bcf20', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current pyspark.ml workflow
2026/03/13 00:12:29 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/opt/conda/lib/python3.11/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <http

🏃 View run enchanting-finch-782 at: http://localhost:5000/#/experiments/137207360586536456/runs/176ea4f71d37428f9308395d45ffea2a
🧪 View experiment at: http://localhost:5000/#/experiments/137207360586536456
🏃 View run painted-doe-907 at: http://localhost:5000/#/experiments/137207360586536456/runs/d6518a0e3fe04e209e06c9e157e8bed4
🧪 View experiment at: http://localhost:5000/#/experiments/137207360586536456
🏃 View run learned-whale-370 at: http://localhost:5000/#/experiments/137207360586536456/runs/ffa9b25452ff44ddb0cb94dfd38b3cb2
🧪 View experiment at: http://localhost:5000/#/experiments/137207360586536456
🏃 View run worried-worm-834 at: http://localhost:5000/#/experiments/137207360586536456/runs/11bbfdaa2e9f4175a1c3b0a512a485ca
🧪 View experiment at: http://localhost:5000/#/experiments/137207360586536456
🏃 View run smiling-fowl-576 at: http://localhost:5000/#/experiments/137207360586536456/runs/5459478a0c5745efba7db1e8315bcf20
🧪 View experiment at: http://localhost:5000/#/experiments/13720

2026/03/13 00:19:00 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '9e0c9ea034ac4fbd98c63142fdfb7b3f', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current pyspark.ml workflow


🏃 View run fun-sloth-653 at: http://localhost:5000/#/experiments/137207360586536456/runs/74caff1d428a46ebb9273a391190710d
🧪 View experiment at: http://localhost:5000/#/experiments/137207360586536456
🏃 View run monumental-shoat-581 at: http://localhost:5000/#/experiments/137207360586536456/runs/f28b6e45d7e649b690c90d610079a590
🧪 View experiment at: http://localhost:5000/#/experiments/137207360586536456
🏃 View run delicate-skink-469 at: http://localhost:5000/#/experiments/137207360586536456/runs/557cffe547df410b9a668e1030810076
🧪 View experiment at: http://localhost:5000/#/experiments/137207360586536456
🏃 View run debonair-deer-218 at: http://localhost:5000/#/experiments/137207360586536456/runs/05ed2b7a910d488d9899b840590e2741
🧪 View experiment at: http://localhost:5000/#/experiments/137207360586536456
🏃 View run salty-duck-477 at: http://localhost:5000/#/experiments/137207360586536456/runs/9e0c9ea034ac4fbd98c63142fdfb7b3f
🧪 View experiment at: http://localhost:5000/#/experiments/1372073

2026/03/13 00:27:52 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID 'dd4e09a577c8482b9dec18cc8d9dc267', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current pyspark.ml workflow


🏃 View run fearless-asp-404 at: http://localhost:5000/#/experiments/137207360586536456/runs/cdda637200b944d9b14df6252c5be27d
🧪 View experiment at: http://localhost:5000/#/experiments/137207360586536456
🏃 View run invincible-fish-835 at: http://localhost:5000/#/experiments/137207360586536456/runs/08871a034ad54d3b993dbe11a79d239a
🧪 View experiment at: http://localhost:5000/#/experiments/137207360586536456
🏃 View run incongruous-shad-912 at: http://localhost:5000/#/experiments/137207360586536456/runs/31917d68025a451ebdb012d1db8cfd7d
🧪 View experiment at: http://localhost:5000/#/experiments/137207360586536456
🏃 View run wise-slug-132 at: http://localhost:5000/#/experiments/137207360586536456/runs/937ba9ca01a343f9b619cc204dd41ca5
🧪 View experiment at: http://localhost:5000/#/experiments/137207360586536456
🏃 View run beautiful-swan-549 at: http://localhost:5000/#/experiments/137207360586536456/runs/dd4e09a577c8482b9dec18cc8d9dc267
🧪 View experiment at: http://localhost:5000/#/experiments/137

2026/03/13 00:32:25 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID 'deb47f6f9a174850a4d768e3aef82571', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current pyspark.ml workflow


----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 35960)
Traceback (most recent call last):
  File "/opt/conda/lib/python3.11/socketserver.py", line 317, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/opt/conda/lib/python3.11/socketserver.py", line 348, in process_request
    self.finish_request(request, client_address)
  File "/opt/conda/lib/python3.11/socketserver.py", line 361, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/opt/conda/lib/python3.11/socketserver.py", line 755, in __init__
    self.handle()
  File "/usr/local/spark/python/pyspark/accumulators.py", line 295, in handle
    poll(accum_updates)
  File "/usr/local/spark/python/pyspark/accumulators.py", line 267, in poll
    if self.rfile in r and func():
                           ^^^^^^
  File "/usr/local/spark/python/pyspark/accumulators.py", line 271, in accum_updates
    num_updates =

🏃 View run shivering-croc-340 at: http://localhost:5000/#/experiments/137207360586536456/runs/deb47f6f9a174850a4d768e3aef82571
🧪 View experiment at: http://localhost:5000/#/experiments/137207360586536456


KeyboardInterrupt: 